# 28. 스태킹 앙상블과 OOF 메타 학습 (Stacking)

서로 다른 기본 모델(LogisticRegression, DecisionTree, SVC)의 교차검증 예측(OOF)을 입력으로 받아 메타 모델이 최종 예측을 수행하는 `StackingClassifier`와 수동 OOF 스태킹을 구현합니다.

In [ ]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import StackingClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.2, random_state=42, stratify=iris.target
)
estimators = [
    ('lr', Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression(max_iter=500))])),
    ('dt', DecisionTreeClassifier(random_state=42)),
    ('svc', Pipeline([('scaler', StandardScaler()), ('clf', SVC(probability=True, random_state=42))])),
]

## 문제 1. StackingClassifier 학습 및 혼동 행렬 확인

`estimators`와 메타 모델 `LogisticRegression(max_iter=1000)`을 `StackingClassifier(cv=5, stack_method="predict_proba")`로 결합해 테스트 정확도와 혼동 행렬을 출력하세요.

In [ ]:
stack_clf = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(max_iter=1000),
    cv=5,
    stack_method='predict_proba',
)
stack_clf.fit(X_train, y_train)
pred = stack_clf.predict(X_test)
print(f'Stacking Accuracy: {accuracy_score(y_test, pred):.4f}')
print(confusion_matrix(y_test, pred))

**결과 해석**

5-fold 교차검증 예측 확률을 메타 특성으로 사용하는 `StackingClassifier`는 훈련 데이터 과적합 없이 다양한 기반 모델의 강점을 결합합니다.

## 문제 2. cross_val_predict 기반 수동 OOF 메타 특성 생성 및 평가

각 기반 모델의 5-fold `cross_val_predict(..., method="predict_proba")`로 `X_meta_train`을 만들고 테스트 예측 확률로 `X_meta_test`를 만들어 메타 로지스틱 회귀의 정확도를 출력하세요.

In [ ]:
oof_list, test_list = [], []
for _, m in estimators:
    oof_list.append(cross_val_predict(m, X_train, y_train, cv=5, method='predict_proba'))
    m.fit(X_train, y_train)
    test_list.append(m.predict_proba(X_test))

X_meta_train = np.hstack(oof_list)
X_meta_test = np.hstack(test_list)
meta = LogisticRegression(max_iter=1000).fit(X_meta_train, y_train)
print(f'Manual OOF Accuracy: {accuracy_score(y_test, meta.predict(X_meta_test)):.4f}')

**결과 해석**

직접 `cross_val_predict`로 Out-Of-Fold 확률을 생성해 메타 모델을 학습하면 `StackingClassifier`와 동일한 누수 방지 원리로 스태킹을 구현할 수 있습니다.